In [ ]:
%%bash
if [ -d "./cifar10_data/cifar-10-batches-py" ] && [ -f "./cifar10_data/cifar-10-batches-py/batches.meta" ]; then
    echo "Dataset already exists, skipping download."
else
    echo "Dataset not found, downloading..."
    kaggle datasets download -d pankrzysiu/cifar10-python --force
    unzip -q cifar10-python.zip -d ./cifar10_data
fi

In [ ]:
import pickle
import numpy as np
import os


def unpickle(file):
    with open(file, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')
    return data_dict


data_dir = './cifar10_data/cifar-10-batches-py/'

x_train_list = []
y_train_list = []

for i in range(1, 6):
    batch_path = os.path.join(data_dir, f'data_batch_{i}')
    batch = unpickle(batch_path)
    x_train_list.append(batch[b'data'])
    y_train_list.append(batch[b'labels'])

x_train = np.concatenate(x_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

test_batch = unpickle(os.path.join(data_dir, 'test_batch'))
x_test = test_batch[b'data']
y_test = np.array(test_batch[b'labels'])

meta_batch = unpickle(os.path.join(data_dir, 'batches.meta'))
classes = [b.decode('utf-8') for b in meta_batch[b'label_names']]

print(f"Training features shape: {x_train.shape}")
print(f"Training labels shape:   {y_train.shape}")
print(f"Testing features shape:  {x_test.shape}")
print(f"Dataset classes:         {classes}")

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

x_train = x_train.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
x_test = x_test.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

x_train_sub = x_train[:1000]
y_train_sub = y_train[:1000]
x_val_sub = x_train[1000:1200]
y_val_sub = y_train[1000:1200]
x_test_sub = x_test[:200]
y_test_sub = y_test[:200]

x_train_resized = tf.image.resize(x_train_sub, (224, 224)).numpy()
x_val_resized = tf.image.resize(x_val_sub, (224, 224)).numpy()
x_test_resized = tf.image.resize(x_test_sub, (224, 224)).numpy()

print("Original shape:", x_train.shape)
print("Resized training shape:", x_train_resized.shape)

plt.figure(figsize=(2, 2))
plt.imshow(x_train_resized[0])
plt.title(classes[y_train_sub[0]])
plt.axis('off')
plt.show()

### Why Resizing is Required
Pretrained models (like VGG16, ResNet50, etc.) were trained on ImageNet, which contains images of size 224x224 pixels. These models have fixed architecture details in their fully connected layers (or benefit from specific spatial dimensions in intermediate layers). Resizing the low-resolution 32x32 CIFAR-10 images to 224x224 allows us to feed them directly into the pretrained networks and leverage the spatial hierarchy and features learned on the larger ImageNet dataset.

### Transfer Learning Overview
- **Transfer Learning**: A machine learning technique where a model developed for a task is reused as the starting point for a model on a second task. It leverages the pre-trained weights from large datasets (like ImageNet) to solve target tasks with limited data.
- **Pretrained Models**:
  - **VGG**: An architecture characterized by its simplicity, using blocks of 3x3 convolution layers followed by max pooling, ending with fully connected layers.
  - **ResNet (Residual Networks)**: Introduces skip connections (residual blocks) to allow training of very deep networks by mitigating the vanishing gradient problem.
- **Advantages**:
  - Much faster training convergence.
  - Requires significantly less labeled data to achieve high performance.
  - Reduces computational cost.

In [ ]:
base_model = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history_frozen = model.fit(x_train_resized, y_train_sub, epochs=5, batch_size=32, validation_data=(x_val_resized, y_val_sub))

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

history_finetune = model.fit(x_train_resized, y_train_sub, epochs=5, batch_size=32, validation_data=(x_val_resized, y_val_sub))

In [ ]:
test_loss_frozen, test_acc_frozen = model.evaluate(x_test_resized, y_test_sub, verbose=0)
print(f"Test Accuracy (Frozen Base): {history_frozen.history['val_accuracy'][-1]:.4f}")
print(f"Test Accuracy (Fine-Tuned): {history_finetune.history['val_accuracy'][-1]:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_frozen.history['accuracy'] + history_finetune.history['accuracy'], label='Train Accuracy')
plt.plot(history_frozen.history['val_accuracy'] + history_finetune.history['val_accuracy'], label='Val Accuracy')
plt.axvline(x=4, color='r', linestyle='--', label='Fine-tuning Start')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_frozen.history['loss'] + history_finetune.history['loss'], label='Train Loss')
plt.plot(history_frozen.history['val_loss'] + history_finetune.history['val_loss'], label='Val Loss')
plt.axvline(x=4, color='r', linestyle='--', label='Fine-tuning Start')
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
y_pred_probs = model.predict(x_test_resized)
y_pred = np.argmax(y_pred_probs, axis=1)

plt.figure(figsize=(10, 10))
plt.suptitle("Sample Predictions")
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(x_test_resized[i])
    pred_label = classes[y_pred[i]]
    true_label = classes[y_test_sub[i]]
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}", color='green' if y_pred[i] == y_test_sub[i] else 'red')
    plt.axis('off')
plt.show()

### Object Detection Basics
- **Object Detection**: The process of identifying and localizing one or more objects of interest within an image.
- **Classification vs. Detection**:
  - **Classification**: Focuses solely on identifying the main object category or class present in the entire image.
  - **Detection**: Classifies multiple objects in the image and locates them by drawing bounding boxes around them.
- **Bounding Boxes**: Coordinate values (usually representing the top-left corner `[x_min, y_min]`, width, and height, or `[x_min, y_min, x_max, y_max]`) that outline the spatial boundaries of detected objects.

### Image Segmentation & U-Net
- **Image Segmentation**: Partitioning an image into multiple segments (sets of pixels) to locate objects and boundaries at the pixel level. There are two main types: semantic segmentation (classifying each pixel) and instance segmentation (distinguishing individual objects).
- **U-Net Architecture**:
  - **Encoder (Contracting Path)**: Captures context and extracts high-level features through downsampling (convolution and pooling layers).
  - **Decoder (Expanding Path)**: Enables precise localization by upsampling the features back to the original resolution.
  - **Skip Connections**: Connect the encoder feature maps directly to the decoder feature maps to retain spatial and structural details.
- **Medical Imaging Applications**: Used for pixel-level task boundaries like organ segmentation, brain tumor localization, vessel segmentation, and lesion delineation from MRI, CT, or X-ray scans.